# ML-03 — Frame Your Lane as an ML Task

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is Refresh/Content Opportunity Scoring. This is a Ranking/Scoring task because the goal is not just to decide yes/no, but instead to give an opportunity score to each content page, in order to rank pages in priority order. The SEO editor will refresh the page with the highest opportunity score first.

## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

I will predict **'is_declining_label'**,whether a content page is declining
in performance or not. This label comes from **'trend_direction'**, where a
value of **"down"** means the page is declining (label = 1), and other values
(stable, up, flat, new) mean it is not declining (label = 0).

This is a defined-rule label rather than a directly observed outcome,
it is derived from **'trend_pct'**, which measures the percentage change in
performance over time. Because the label is based on **'trend_direction' **
and **'trend_pct'**, these two columns will never be used as features, to
avoid leakage, the model would otherwise just learn the rule that
created the label instead of learning real patterns.

## 3. Success metric

*One metric you can defend. What number means 'good'?*

I'll use **Precision@20** as my metric, meaning, out of the top 20 pages
my ranking flags for review, how many are actually declining?

Looking at the data, 54.2% of all pages are already labeled as declining.
That's important, because it sets my baseline — if I picked 20 pages at
random, I'd expect around 54% of them to be declining just by chance. So
for my ranking to actually mean something, its Precision@20 needs to be
noticeably higher than 0.542. Anything close to that number would just
mean the ranking isn't doing better than a random guess.

I picked this metric because it mirrors how the output would actually be
used, an SEO editor isn't going to review all 30,000 pages, they'll only
have time for a short list. So what really matters is whether the pages
at the top of that list are the ones that genuinely need attention.

## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

In [4]:
!git clone https://github.com/ainasarfaraz343-a11y/flyrank-internship.git

Cloning into 'flyrank-internship'...
remote: Enumerating objects: 112, done.
remote: Counting objects: 100% (112/112), done.
remote: Compressing objects: 100% (83/83), done.
remote: Total 112 (delta 29), reused 78 (delta 13), pack-reused 0 (from 0)
Receiving objects: 100% (112/112), 1.84 MiB | 9.29 MiB/s, done.
Resolving deltas: 100% (29/29), done.


In [5]:
%cd flyrank-internship

/content/flyrank-internship


In [9]:
import pandas as pd

# Load full dataset
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

#From trend_direction making declining
df['is_declining_label'] = (df['trend_direction'] == 'down').astype(int)

# check how many pages are declining
print(df['is_declining_label'].value_counts())

is_declining_label
1    16262
0    13738
Name: count, dtype: int64


In [10]:
lane_cols = ["content_id", "client_id", "content_type", "avg_position",
             "ctr", "content_age_days", "freshness_tier",
             "trend_direction", "is_declining_label"]

df_lane = df[lane_cols]
df_lane.head(10)

,content_id,client_id,content_type,avg_position,ctr,content_age_days,freshness_tier,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,10.6,0.76,187,0-30,down,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,20.3,0.05,445,0-30,down,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,36.5,0.09,141,0-30,down,1
3,content_331d6c4de07b,client_19581e27de,keyword article,6.2,0.49,463,0-30,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,44.0,0.13,263,0-30,down,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,8.5,0.03,147,0-30,down,1
6,content_9a34b442b552,client_8722616204,keyword article,7.0,0.00,90,0-30,down,1
7,content_a63219c6e95a,client_19581e27de,keyword article,21.2,0.06,445,0-30,stable,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,46.0,0.09,90,0-30,down,1
9,content_c27558df2b0c,client_19581e27de,keyword article,4.9,0.16,257,91-180,down,1


One row= one content page. Every row represent an individual content page belonging to a specific client, along with its performance metrics
(position, CTR, age, freshness) and its trend status.The is_declining_label is derived from trend_direction (where 'down'= declining) and this
is the target/proxy for scoring refresh opportunity.
Note: 'trend_direction' and 'trend_pct' are never used as features, only as the source for the label, to avoid leakage.

## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A simple rule like "if avg_position > 20, refresh the page" only looks at
one thing at a time, but priority isn't really that simple. A page's age,
its trend direction, its CTR, and how much visibility it still has all
matter together, not separately. For example, a page could have a bad
position but still be brand new and improving — that's not urgent. Another
page might have a decent position but be old, stale, and quietly losing
traffic — that one probably is urgent. A single if-statement just can't
tell those two cases apart.

To handle this properly with fixed rules, I'd basically need a huge list
of hand-picked thresholds covering every combination of signals, and even
then those thresholds would probably need to change from client to client
or content type to content type. That's a lot of guesswork and constant
tuning. This is exactly the kind of messy, multi-signal pattern that's
better learned from the data itself through a scoring approach, rather
than hard-coded by hand one condition at a time.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.